# Minimal config

The shortest path from "I have a melt composition" to "I have a saturation pressure for it." Everything you don't set falls through to volcatenate's defaults — the same defaults the [Sulfur Comparison Paper](https://github.com/kaylai/volcatenate) uses.

If you want to know what each default actually does inside the model, the [config propagation reference](../config_propagation.md) walks every YAML field through to its underlying backend call. This notebook shows you the smallest possible runnable example; the [full-config notebook](full_config.ipynb) shows what every option looks like when set explicitly.

## 1. Build a sample composition

A `MeltComposition` is a single melt-inclusion / sample. Oxides, volatiles, and a redox indicator. Anything you don't supply defaults to 0 (or `None` for the optional redox columns).

In [ ]:
from volcatenate.composition import MeltComposition

fuego = MeltComposition(
    sample="Fuego",
    T_C=1030,
    SiO2=53.7, TiO2=1.11, Al2O3=18.2, FeOT=9.83,
    MnO=0.20, MgO=3.94, CaO=8.34, Na2O=3.62, K2O=0.81, P2O5=0.25,
    H2O=4.7, CO2=0.085, S=0.028,
    Fe3FeT=0.24,
)
fuego

## 2. Default config

`RunConfig()` constructs every backend's sub-config with paper defaults. You can pass it as-is to any of the calculator functions.

In [ ]:
from volcatenate.config import RunConfig

config = RunConfig()
print("output dir:", config.output_dir)
print("verbose:  ", config.verbose)
print("EVo gas system:", config.evo.gas_system)
print("VolFe gassing style:", config.volfe.gassing_style)
print("MAGEC redox option:", config.magec.redox_option)

## 3. Override one or two settings

Most users tune just a few fields. You can do this two ways: edit the dataclass directly, or load a YAML file. Either is fine; they share the same shape.

In [ ]:
config = RunConfig()
config.output_dir = "my_run_output"
config.verbose = True
config.evo.p_start = 5000          # bar (paper default is 3000)
config.magec.timeout = 600         # seconds

# Save to YAML so you can re-use the same settings later:
from volcatenate.config import save_config
save_config(config, "my_run_config.yaml")

## 4. Run a saturation-pressure calculation

Pick whichever backends you have installed. The cell below tries EVo and VolFe; skip it if you don't have those packages on your machine. The output is a tidy DataFrame with one row per sample and one column per model.

In [ ]:
from volcatenate.core import calculate_saturation_pressure

# Pick whichever backends are available in your environment.
result = calculate_saturation_pressure(
    [fuego],
    models=["EVo", "VolFe"],
    config=config,
)
result.pressure

## What's next

- The [full-config notebook](full_config.ipynb) walks every backend's config one section at a time, showing what each option does and what it's set to by default.
- The [config propagation reference](../config_propagation.md) explains, in plain English, what each YAML field actually does once it reaches the underlying model — including which values are silently pulled from the sample composition and what fallback chains apply when redox data is missing.
- The [run-bundles guide](../run_bundles.md) shows how to capture a fully-reproducible JSON record of a run, including the exact config and the resolved per-(sample, backend) inputs each model received.